In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
base_path = r"/storage/alplakes_test/neuchatel_100m_2025"
input_folder = os.path.join(base_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(base_path, "outputs_swirl", "ke_eddy")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
lvl0_csv_path = os.path.join(input_folder, "lvl0.csv")
lake_csv_path = os.path.join(input_folder, "lake_characteristics.csv")

In [ ]:
df_lvl0 = pd.read_csv(lvl0_csv_path)
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])

In [ ]:
df_lake = pd.read_csv(lake_csv_path)
df_lake = df_lake.set_index('id', drop=False)
df_lake['date'] = pd.to_datetime(df_lake['date'])

# Entire lake analysis

In [ ]:
eddy_ke = df_lvl0.groupby('date').sum()['kinetic_energy_eddy_[MJ]']

In [ ]:
lake_ke = df_lake.groupby('date').sum()['kinetic_energy_[MJ]']

In [ ]:
eddy_ke.plot()

In [ ]:
eddy_ke.reset_index().to_csv(os.path.join(output_folder, "ke_eddies.csv"))
lake_ke.reset_index().to_csv(os.path.join(output_folder, "ke_lake.csv"))

In [ ]:
eddy_ke = pd.read_csv(os.path.join(output_folder, "ke_eddies.csv"), index_col=0)
lake_ke = pd.read_csv(os.path.join(output_folder, "ke_lake.csv"), index_col=0)
eddy_ke['date'] = pd.to_datetime(eddy_ke['date'])
lake_ke['date'] = pd.to_datetime(lake_ke['date'])
eddy_ke = eddy_ke.set_index('date')
lake_ke = lake_ke.set_index('date')

In [ ]:
import matplotlib.dates as mdates

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
lake_ke.plot(ax=ax, color='tab:orange')
eddy_ke.plot(ax=ax, color='tab:blue')
plt.ylabel('Kinetic Energy [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
plt.grid(False)
ax.legend(['Entire lake', 'Eddies'])
fig.tight_layout()
fig.savefig(os.path.join(output_folder, "kinetic_energy.png"))

In [ ]:
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import (
    ColumnDataSource, HoverTool,
    Range1d, LinearAxis
)
output_notebook()

In [ ]:
# --- Prepare data sources ---
src = ColumnDataSource(data=dict(
    date=lake_ke.index,
    lake_ke=lake_ke.values,
    eddy_ke=eddy_ke.values
))

# --- Create the base figure (left axis) ---
p = figure(
    title=f'Kinetic Energy',
    x_axis_type='datetime',
    width=1200,
    height=450,
    x_axis_label='Date',
    y_axis_label='KE (MJ)',
    tools='xpan,xwheel_zoom,box_zoom,reset,save',
    active_scroll='xwheel_zoom',
)

# Determine the ranges
left_min = 0
left_max = 100000

# Set left y-range explicitly
p.y_range = Range1d(start=left_min, end=left_max)


# Plot left axis line
p.line(
    x='date', y='eddy_ke',
    source=src,
    legend_label='eddy_ke',
    line_color='#E45756',
    line_width=1.5,
    alpha=0.8
)

# Plot right axis line
p.line(
    x='date', y='lake_ke',
    source=src,
    legend_label='lake_ke',
    line_width=1.5,
    alpha=0.8,
)

# Hover (now works properly)
p.add_tools(HoverTool(
    tooltips=[
        ("Date", "@date{%F %H:%M}"),
        ("Dissipation (MJ/h)", "@dissipation{0.00}"),
        ("KE upper layer (MJ)", "@ke{0.00}")
    ],
    formatters={"@date": "datetime"},
    mode="vline"
))

p.legend.location = 'top_right'
p.legend.click_policy = 'hide'

show(p)


In [ ]:
fig = plt.figure(figsize=(10, 5))
ax1 = fig.add_subplot(1, 1, 1)

# Left axis: lake KE
lake_ke.plot(ax=ax1)
ax1.set_ylabel('Lake Kinetic Energy [MJ]')

# Right axis: eddy KE
ax2 = ax1.twinx()
eddy_ke.plot(ax=ax2, color='darkorange')
ax2.set_ylabel('Eddy Kinetic Energy [MJ]')

# Shared x-axis formatting
ax1.set_xlabel('')
plt.xticks(rotation=10)

# Legend (manual, since two axes)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, ['Entire lake', 'Eddies'])

ax1.grid(True)
ax2.grid(False)

fig.savefig(os.path.join(output_folder, "kinetic_energy_2axes.png"))
plt.show()

In [ ]:
percentage_total_ke = pd.DataFrame({'date': lake_ke.index, 'perc_ke': (100 * eddy_ke.values / lake_ke[lake_ke.index.isin(eddy_ke.index)].values).flatten()})
percentage_total_ke = percentage_total_ke.set_index('date')

In [ ]:
percentage_total_ke.mean().values

In [ ]:
percentage_total_ke.plot(color='tab:green', legend=False, figsize=(10,5))
plt.hlines(percentage_total_ke.mean(),
           percentage_total_ke.index[0],
           percentage_total_ke.index[-1],
           linestyles='dashed',
           color='tab:blue')

plt.text(0.98,0.02,f'Mean={percentage_total_ke.values.mean():.1f}%', transform=plt.gca().transAxes, ha='right', va='bottom')

plt.title('Fraction of total kinetic energy contained in eddies')
plt.ylabel('Fraction [%]')
plt.xlabel('')
plt.xticks(rotation=10)
plt.grid(False)

plt.savefig(os.path.join(output_folder, "fraction_ke.png"))

# Depth analysis

In [ ]:
df_lvl0['layer_thickness_[m]'] = df_lvl0['volume_slice_[m3]'] / df_lvl0['surface_area_[m2]']
df_lake['layer_thickness_[m]'] = df_lake['volume_slice_[m3]'] / df_lake['surface_area_[m2]']

In [ ]:
df_lvl0['ke_per_meter_[MJ/m]'] = df_lvl0['kinetic_energy_eddy_[MJ]'] / df_lvl0['layer_thickness_[m]']
df_lake['ke_per_meter_[MJ/m]'] = df_lake['kinetic_energy_[MJ]'] / df_lake['layer_thickness_[m]']

### Time-Depth plots

In [ ]:
profile_eddy_ke = df_lvl0.groupby(['date', 'depth_[m]', 'time_index'])[['ke_per_meter_[MJ/m]']].sum().reset_index()

In [ ]:
profile_eddy_ke.head()

In [ ]:
pivot_profile_eddy_ke = profile_eddy_ke.pivot(index="depth_[m]", columns="date", values="ke_per_meter_[MJ/m]")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_profile_eddy_ke.columns,
    pivot_profile_eddy_ke.index,
    pivot_profile_eddy_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Kinetic Energy contained in eddies")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Kinetic energy [MJ]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ke_eddy_time_depth.png"))
plt.show()

In [ ]:
pivot_lake_ke = df_lake.pivot(index="depth_[m]", columns="date", values="ke_per_meter_[MJ/m]")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_lake_ke.columns,
    pivot_lake_ke.index,
    pivot_lake_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Total Kinetic Energy")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Kinetic energy [MJ]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ke_total_time_depth.png"))
plt.show()

In [ ]:
pivot_ratio_ke = 100 * pivot_profile_eddy_ke / pivot_lake_ke

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

pcm = ax.pcolormesh(
    pivot_ratio_ke.columns,
    pivot_ratio_ke.index,
    pivot_ratio_ke.values,
    shading="auto",
    cmap="viridis"
)

# Labels
ax.set_xlabel("Time")
ax.set_ylabel("Depth")
ax.set_title("Ratio Kinetic Energy in eddies")

# Colorbar
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label("Ratio of Kinetic Energy [%]")

plt.tight_layout()
plt.savefig(os.path.join(output_folder, "ratio_ke_time_depth.png"))
plt.show()

### KE profiles

In [ ]:
os.makedirs(os.path.join(output_folder, "profiles_ke"), exist_ok=True)
for t_idx in range(1,8760, 24):
    plt.close('all')
    fig, ax = plt.subplots(figsize=(5,7))

    # Lake KE profile
    df_lake[df_lake['time_index']==t_idx].plot(
        x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Lake KE'
    )

    # Eddy KE profile
    profile_eddy_ke[profile_eddy_ke['time_index']==t_idx].plot(
        x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Eddy KE'
    )

    ax.set_xlabel("Kinetic Energy [MJ/m]")
    ax.set_ylabel("Depth [m]")
    ax.legend()
    plt.title(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0])
    ax.set_xlim(left=0, right=280)
    #plt.show()
    str_time = str(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0]).replace(' ','_').replace(':','-')
    fig.savefig(os.path.join(output_folder, "profiles_ke", f"kinetic_energy_profile_{str_time}.png"))

In [ ]:
fig, ax = plt.subplots(figsize=(5,7))
profile_eddy_ke[profile_eddy_ke['time_index']==t_idx].plot(
    x='ke_per_meter_[MJ/m]', y='depth_[m]', ax=ax, label='Eddy KE'
)
plt.title(df_lake[df_lake['time_index']==t_idx]['date'].iloc[0])
plt.show()

# Specific depth analysis

In [ ]:
depths = pd.read_csv(os.path.join(base_path, "grid", "depths.csv"))

In [ ]:
i_depth_min = 0
i_depth_max = len(depths)-1

In [ ]:
def filter_by_depths(df, i_depth_min, i_depth_max):
    depth_filter = (
            (df['depth_index'] <= i_depth_max) &
            (df['depth_index'] >= i_depth_min)
        )
    return df[depth_filter]

In [ ]:
str_depth_min = str(round(depths.iloc[i_depth_min]['depth_[m]'], 2))
str_depth_max = str(round(depths.iloc[i_depth_max]['depth_[m]'], 2))

print(str_depth_min, str_depth_max)

In [ ]:
df_lvl0_filtered_by_depth = filter_by_depths(df_lvl0, i_depth_min, i_depth_max).set_index('time_index', drop=False)
df_lake_filtered_by_depth = filter_by_depths(df_lake, i_depth_min, i_depth_max).set_index('time_index', drop=False)

In [ ]:
lvl0_filtered_by_depth_sum = df_lvl0_filtered_by_depth.groupby('date').sum()
lake_filtered_by_depth_sum = df_lake_filtered_by_depth.groupby('date').sum()

In [ ]:
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].reset_index().to_csv(os.path.join(output_folder, f"ke_eddies_{str_depth_min}-{str_depth_max}m.csv"))
lake_filtered_by_depth_sum['kinetic_energy_[MJ]'].reset_index().to_csv(os.path.join(output_folder, f"ke_lake_{str_depth_min}-{str_depth_max}m.csv"))

In [ ]:
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'] = 100 * lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'] / lake_filtered_by_depth_sum['kinetic_energy_[MJ]']

In [ ]:
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'].plot(label=f'{str_depth_min}-{str_depth_max}m')

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
percentage_total_ke.plot(label='Entire lake')
lvl0_filtered_by_depth_sum['percentage_total_ke_[%]'].plot(label=f'{str_depth_min}-{str_depth_max}m')
plt.title('Fraction of total kinetic energy contained in eddies')
plt.ylabel('Fraction [%]')
plt.xlabel('')
plt.xticks(rotation=10)
plt.legend()
fig.savefig(os.path.join(output_folder, f"fraction_ke_{str_depth_min}-{str_depth_max}m.png"))

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
lake_filtered_by_depth_sum['kinetic_energy_[MJ]'].plot(label='Total')
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].plot(label='Eddies')
plt.legend()
plt.ylabel(f'Kinetic Energy {str_depth_min}-{str_depth_max}m [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
#ax.set_yscale('log')
fig.savefig(os.path.join(output_folder, f"kinetic_energy_{str_depth_min}-{str_depth_max}m.png"))

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 1, 1)
eddy_ke.plot(label='Entire depth')
lvl0_filtered_by_depth_sum['kinetic_energy_eddy_[MJ]'].plot(label=f'{str_depth_min}-{str_depth_max}m')
plt.legend()
plt.title('Kinetic Energy contained in eddies')
plt.ylabel(f'Kinetic Energy [MJ]')
plt.xlabel('')
plt.xticks(rotation=10)
fig.savefig(os.path.join(output_folder, f"eddy_kinetic_energy_{str_depth_min}-{str_depth_max}m.png"))

# Horizontal distribution

In [ ]:
da_ke = xr.open_dataarray(os.path.join(base_path, 'energy_budget', 'kinetic_energy.nc'))

In [ ]:
da_eddy_mask = xr.open_dataarray(os.path.join(base_path, 'outputs_swirl', 'eddy_catalogues_final', 'eddy_mask.nc'))

In [ ]:
import dask.array as da

da_ke = da_ke.chunk({"time": 24, "Z": 1, "YC": 180, "XC": 288})
da_eddy_mask = da_eddy_mask.chunk({"time": 24, "Z": 1, "YC": 180, "XC": 288})


In [ ]:
da_ke_horizontal_sum = da_ke.sum(dim=['time'])
da_ke_horizontal_sum.to_netcdf(os.path.join(base_path, 'energy_budget', 'sum_ke.nc'))

In [ ]:
da_ke_eddy = da_ke.where(da_eddy_mask)

In [ ]:
da_ke_eddy_horizontal_sum = da_ke_eddy.sum(dim=['time'])

In [ ]:
da_ke_eddy_horizontal_sum.to_netcdf(os.path.join(base_path, 'energy_budget', 'sum_ke_eddy.nc'))

In [ ]:
da_ke_eddy_horizontal_sum = xr.open_dataarray(os.path.join(base_path, 'energy_budget', 'sum_ke_eddy.nc'))

In [ ]:
import numpy as np

In [ ]:
plt.figure(figsize=(10,5))
horizontal_sum = da_ke_eddy_horizontal_sum.sum(dim=['Z'])
horizontal_sum.where(horizontal_sum>0, np.nan).plot(cmap='jet', cbar_kwargs={'label': 'Kinetic Energy [J]'})
plt.title('Annual Vertically Integrated Eddy Kinetic Energy')